In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers accelerate bitsandbytes sentencepiece huggingface_hub datasets evaluate rouge_score bert_score

In [ ]:
import gc
import json
import random
import re
from pathlib import Path

import torch
import evaluate
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# HF Token
login("hf_...")

In [3]:
# Config
SEED = 42
random.seed(SEED)

HF_TOKEN = "YOUR_HF_TOKEN"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DATA_PATH = "/kaggle/input/datasets/hartzbyte/scientific-research-papers/dataset.json"

WORKDIR = Path("/kaggle/working/baseline_eval")
WORKDIR.mkdir(parents=True, exist_ok=True)

MAX_TEST_SAMPLES = 50
MAX_NEW_TOKENS = 512

In [4]:
# Inspect mounted Kaggle inputs
print("Mounted input files:")
for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(root, files[:10])

Mounted input files:
/kaggle/input/datasets/hartzbyte/scientific-research-papers ['dataset.json']


In [5]:
# Load dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))
random.shuffle(data)

Total samples: 1500


In [6]:
# Train/val/test split
n = len(data)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]
test_subset = test_data[:MAX_TEST_SAMPLES]

print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(test_data)}")
print(f"Baseline subset: {len(test_subset)}")

for name, split in [
    ("train.json", train_data),
    ("val.json", val_data),
    ("test.json", test_data),
]:
    with open(WORKDIR / name, "w", encoding="utf-8") as f:
        json.dump(split, f, indent=2, ensure_ascii=False)

Train: 1200
Val: 150
Test: 150
Baseline subset: 50


In [7]:
# Helper functions
def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def flatten_output(output_obj):
    parts = []
    for key, value in output_obj.items():
        parts.append(f"{key}:")
        if isinstance(value, list):
            for item in value:
                parts.append(f"- {normalize_ws(str(item))}")
        else:
            parts.append(normalize_ws(str(value)))
    return "\n".join(parts)

def expected_fields_from_output(output_obj):
    return list(output_obj.keys())

def build_prompt(instruction, abstract, expected_fields):
    field_lines = "\n".join([f"- {field}" for field in expected_fields])
    return f"""[INST]
You are analyzing a research paper abstract.

Task:
{instruction}

Return the answer using only the following fields:
{field_lines}

Keep the response faithful to the abstract.
Do not invent unsupported claims.

Research Abstract:
{abstract}
[/INST]"""

def extract_assistant_response(decoded_text, prompt):
    if decoded_text.startswith(prompt):
        return decoded_text[len(prompt):].strip()
    return decoded_text.strip()

def schema_field_presence(text, expected_fields):
    text_lower = text.lower()
    return {field: f"{field.lower()}:" in text_lower for field in expected_fields}

In [8]:
# Load tokenizer + model directly in 4-bit
# IMPORTANT: never load full precision first
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
print("Model loaded successfully.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully.


In [9]:
# Smoke test
sample_item = test_subset[0]
sample_fields = expected_fields_from_output(sample_item["output"])
sample_prompt = build_prompt(sample_item["instruction"], sample_item["input"], sample_fields)

inputs = tokenizer(
    sample_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n=== Smoke Test ===\n")
print(extract_assistant_response(decoded, sample_prompt)[:1500])


=== Smoke Test ===

summary: This paper presents a Bayesian learning model to understand the behavior of Large Language Models (LLMs) in terms of their optimization metric of next token prediction. The authors develop a theoretical framework based on an ideal generative text model and examine how LLMs approximate this model. Key contributions include a continuity theorem linking embeddings to multinomial distributions, demonstrating alignment between LLM text generation and Bayesian learning principles, explaining the emergence of in-context learning in larger models, and validating the findings through visualizations of next token probabilities from an instrumented Llama model.

methods:


In [10]:
# Run baseline generation on test subset
predictions = []
references = []
records = []

for idx, item in enumerate(test_subset):
    instruction = item["instruction"]
    abstract = item["input"]
    reference_obj = item["output"]
    expected_fields = expected_fields_from_output(reference_obj)

    prompt = build_prompt(instruction, abstract, expected_fields)
    reference_text = flatten_output(reference_obj)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction_text = extract_assistant_response(decoded, prompt)
    schema_check = schema_field_presence(prediction_text, expected_fields)

    predictions.append(prediction_text)
    references.append(reference_text)

    records.append({
        "id": idx,
        "instruction": instruction,
        "input": abstract,
        "expected_fields": expected_fields,
        "reference_text": reference_text,
        "prediction_text": prediction_text,
        "schema_check": schema_check
    })

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_subset):
        print(f"Processed {idx + 1}/{len(test_subset)}")

# Save predictions
with open(WORKDIR / "baseline_predictions.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

Processed 10/50
Processed 20/50
Processed 30/50
Processed 40/50
Processed 50/50


In [11]:
# Metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_result = rouge.compute(predictions=predictions, references=references)
bertscore_result = bertscore.compute(predictions=predictions, references=references, lang="en")
avg_bertscore_f1 = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])

total_expected = 0
total_found = 0
exact_schema_match_count = 0

for rec in records:
    checks = rec["schema_check"]
    found_count = sum(1 for v in checks.values() if v)
    expected_count = len(checks)

    total_found += found_count
    total_expected += expected_count

    if found_count == expected_count:
        exact_schema_match_count += 1

schema_field_recall = total_found / total_expected if total_expected else 0.0
exact_schema_match = exact_schema_match_count / len(records) if records else 0.0

metrics = {
    "model_id": MODEL_ID,
    "num_samples": len(records),
    "rouge": rouge_result,
    "avg_bertscore_f1": avg_bertscore_f1,
    "schema_field_recall": schema_field_recall,
    "exact_schema_match": exact_schema_match,
    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False
    }
}

with open(WORKDIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("\n=== Baseline Metrics ===")
print(json.dumps(metrics, indent=2))

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Baseline Metrics ===
{
  "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
  "num_samples": 50,
  "rouge": {
    "rouge1": 0.5695318822440322,
    "rouge2": 0.3576774844770573,
    "rougeL": 0.43849877285630545,
    "rougeLsum": 0.5107423602383044
  },
  "avg_bertscore_f1": 0.9106939399242401,
  "schema_field_recall": 0.9428571428571428,
  "exact_schema_match": 0.88,
  "generation_config": {
    "max_new_tokens": 512,
    "do_sample": false
  }
}


In [12]:
# Save manual review samples
manual_review = []
for rec in records[:10]:
    manual_review.append({
        "instruction": rec["instruction"],
        "expected_fields": rec["expected_fields"],
        "reference_text": rec["reference_text"],
        "prediction_text": rec["prediction_text"]
    })

with open(WORKDIR / "manual_review_samples.json", "w", encoding="utf-8") as f:
    json.dump(manual_review, f, indent=2, ensure_ascii=False)

print(f"\nSaved files in: {WORKDIR}")
print("Files:")
for p in sorted(WORKDIR.iterdir()):
    print("-", p.name)


Saved files in: /kaggle/working/baseline_eval
Files:
- baseline_metrics.json
- baseline_predictions.json
- manual_review_samples.json
- test.json
- train.json
- val.json
